In [1]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [2]:
train_df= pd.read_csv("samsum-train.csv")
val_df= pd.read_csv("samsum-validation.csv")

In [3]:
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_df["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_df.shape

(14732, 3)

In [6]:
val_df.shape

(818, 3)

## Random Sampling

In [7]:
train_df= train_df.sample(n= 4000, random_state= 42,).reset_index(drop= True)
val_df= val_df.sample(n= 500, random_state= 42,).reset_index(drop= True)

In [8]:
train_df.shape

(4000, 3)

## Preprocessing

In [9]:
import re
def clean_text(text):
    text = re.sub(r'\r\n', ' ', text)  # Replace newlines with space
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    text= re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text= text.strip()  # Remove leading and trailing whitespace
    return text


In [10]:
train_df['dialogue']= train_df['dialogue'].apply(clean_text)
train_df['summary']= train_df['summary'].apply(clean_text)

val_df['dialogue']= val_df['dialogue'].apply(clean_text)
val_df['summary']= val_df['summary'].apply(clean_text)


In [11]:
train_df["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:  Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

## Tokenization

In [12]:
tokenizer= T5Tokenizer.from_pretrained("t5-small")


## Raw data => Tokenized inputs for fine tuning

In [13]:
def tokenize(data):
    inputs= tokenizer(data['dialogue'], padding= 'max_length', truncation= True, max_length= 512)
    targets= tokenizer(data['summary'], padding= 'max_length', truncation= True, max_length= 128)
    inputs['labels']= targets['input_ids']
    return inputs

In [14]:
train_df_set= train_df.apply(tokenize, axis= 1).tolist()
val_df_set= val_df.apply(tokenize, axis= 1).tolist()

In [15]:
train_df_set[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Working with Model

In [16]:
model= T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [17]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
Device name: NVIDIA GeForce RTX 3060 Laptop GPU


##  Training

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="epoch",
    warmup_steps=500,
    fp16=True
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_df_set,
    eval_dataset=val_df_set,
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,4.424914,0.456752
2,0.476320,0.428128
3,0.447888,0.420234
4,0.433497,0.416141
5,0.425285,0.415173
6,0.421017,0.414860


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=1.1048200174967449, metrics={'train_runtime': 973.2147, 'train_samples_per_second': 24.661, 'train_steps_per_second': 3.083, 'total_flos': 3248203235328000.0, 'train_loss': 1.1048200174967449, 'epoch': 6.0})

In [21]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [22]:
model= T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [23]:
import torch

# Define CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move your model to the GPU
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## Testing the core logic of the model

In [24]:
# Move model to GPU once outside the function
model.to(device)

def summarize_dialogue(dialogue):
    dialogue = clean_text(dialogue)
    
    # 1. Tokenize input
    inputs = tokenizer(
        dialogue, 
        padding="max_length", 
        truncation=True, 
        max_length=512,
        return_tensors="pt"
    ).to(device)
    
    # 2. Generate summary with mixed precision for RTX 3060 speedup
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            targets = model.generate(
                input_ids=inputs["input_ids"],            # Fixed variable name
                attention_mask=inputs["attention_mask"],  # Fixed variable name
                max_length=128,
                num_beams=4,
                early_stopping=True
            )
    
    # 3. Decode output tokens
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary
    
    

In [ ]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

Summary: Artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.
